In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

In [4]:
import os
os.getcwd()

'/Users/bradlarson'

In [6]:
import os
os.chdir("/Users/bradlarson/Desktop/Applied-Project")


In [12]:
df= pd.read_csv("V2_ML_Water_Usage_Dataset.csv")
df.head()

,Wafer Step,Year,Synthetic_Water_per_Wafer,Synthetic_Water_per_Wafer_noisy,Investment_Level,Wafer_Size,Water_Efficiency_Level,Percent_Reclaimed_Water,Percent_Municipal_Water,Reclamation_Tech_Level,Wafer_Intention,Investment_Efficiency,Water_Stress_Index,Composite_Risk_Score,Year_Squared
0,Cleaning,1,0.000032,0.000034,218.543054,300,Low,82.021890,17.978110,5,Reduced,43.708611,5393.433070,146.106567,1
1,Cleaning,2,0.000033,0.000032,477.821438,300,Low,64.720525,35.279475,4,Reduced,119.455360,10583.842650,140.616157,4
2,Cleaning,3,0.000034,0.000035,379.397274,300,Low,56.316925,43.683075,5,Advanced,75.879455,13104.922450,138.395078,9
3,Cleaning,4,0.000035,0.000035,319.396318,450,Medium,73.515460,26.484540,2,Advanced,159.698159,11918.043170,202.654638,16
4,Cleaning,5,0.000036,0.000035,120.208388,200,Medium,59.200136,40.799864,4,Current,30.052097,8159.972703,98.960041,25


In [14]:
df_encoded = df.copy()
label_cols = ['Wafer Step', 'Water_Efficiency_Level', 'Wafer_Intention']
encoder_dict = {}

In [16]:
for col in label_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoder_dict[col] = le

In [30]:
# Reapply updated defect risk simulation logic
def simulate_defect_risk(row):
    base_risk = 0.5

    if row['Percent_Reclaimed_Water'] < 80:
        base_risk += 0.3
    elif row['Percent_Reclaimed_Water'] < 90:
        base_risk += 0.15
    elif row['Percent_Reclaimed_Water'] < 95:
        base_risk += 0.05
    elif row['Percent_Reclaimed_Water'] >= 99:
        base_risk -= 0.1

    if row['Water_Efficiency_Level'] == 0:
        base_risk += 0.2
    elif row['Water_Efficiency_Level'] == 1:
        base_risk += 0.1
    else:
        base_risk -= 0.05

    if row['Reclamation_Tech_Level'] < 3:
        base_risk += 0.15
    elif row['Reclamation_Tech_Level'] >= 4:
        base_risk -= 0.1

    if row['Investment_Efficiency'] < 75:
        base_risk += 0.15
    elif row['Investment_Efficiency'] > 150:
        base_risk -= 0.1

    return round(min(max(base_risk, 0.0), 1.0), 4)

df_encoded['Defect_Risk_Score'] = df_encoded.apply(simulate_defect_risk, axis=1)


In [34]:
# Constants (can be made dynamic later in dashboard)
cost_per_liter = 1.25  # USD
avg_cost_per_defect = 1000.00  # USD
wafer_output_per_year = 100000  # wafers produced per year
baseline_water_per_wafer = 0.0001  # m³ (100 ml)


In [36]:
df_roi = df_encoded.copy()

# Water savings
df_roi["Water_Savings_per_Wafer"] = baseline_water_per_wafer - df_roi["Synthetic_Water_per_Wafer_noisy"]
df_roi["Water_Savings_per_Year"] = df_roi["Water_Savings_per_Wafer"] * wafer_output_per_year
df_roi["Water_Savings_Dollars"] = df_roi["Water_Savings_per_Year"] * cost_per_liter

# Defect costs
df_roi["Defect_Costs_per_Year"] = df_roi["Defect_Risk_Score"] * wafer_output_per_year * avg_cost_per_defect

# Investment cost (convert $K to $)
df_roi["Investment_Cost"] = df_roi["Investment_Level"] * 1000

# Final ROI calculation
df_roi["Net_Benefit"] = df_roi["Water_Savings_Dollars"] - df_roi["Defect_Costs_per_Year"]
df_roi["ROI"] = df_roi["Net_Benefit"] / df_roi["Investment_Cost"]


In [38]:
roi_preview = df_roi[[
    "Wafer Step", "Investment_Level", "Synthetic_Water_per_Wafer_noisy", "Water_Savings_Dollars",
    "Defect_Risk_Score", "Defect_Costs_per_Year", "Investment_Cost", "Net_Benefit", "ROI"
]]

roi_preview.head()  # Or display in Streamlit/dashboard


,Wafer Step,Investment_Level,Synthetic_Water_per_Wafer_noisy,Water_Savings_Dollars,Defect_Risk_Score,Defect_Costs_per_Year,Investment_Cost,Net_Benefit,ROI
0,1,218.543054,0.000034,8.2875,0.8,80000000.0,218543.0535,-7.999999e+07,-366.060556
1,1,477.821438,0.000032,8.5375,0.8,80000000.0,477821.4379,-7.999999e+07,-167.426543
2,1,379.397274,0.000035,8.1250,0.8,80000000.0,379397.2738,-7.999999e+07,-210.860745
3,1,319.396318,0.000035,8.0875,0.8,80000000.0,319396.3179,-7.999999e+07,-250.472493
4,1,120.208388,0.000035,8.1250,0.8,80000000.0,120208.3882,-7.999999e+07,-665.510894


In [50]:
#Step What It Tells You
#1 Water Savings	Per wafer & annual
#2 Savings ($)	Dollar value of saved water
# 3 Defect Losses	How much defects cost annually
# 4 Investment Cost	What you spent to improve the system
#5 Net Benefit	Water savings − defect losses
# 6 ROI	Profit or loss per $1 invested
# ROI = (Water_Savings_Dollars - Defect_Costs_per_Year) / Investment_Cost
#No ML neede for this its all rules-based. Known variables such as water savings, defect costs, and investment; business assumptions and ROI formula
#There was no need to train another model to estimate ROI - its derived from the two Water per Wafer and Defect Risk ML + some math. 
